In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from transformers import Wav2Vec2Model
from typing import Dict, List

import librosa
import torchaudio
import torchaudio.transforms as T
from torchaudio.models.conformer import Conformer

from accelerate import Accelerator
from accelerate.utils import DistributedDataParallelKwargs


import contextlib
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import ast
from typing import Dict, List
import os

In [2]:
# ? this is for local training

MODEL_PATH = "../../models/best_checkpoint_v5.pth"
WORKING_MODEL_PATH = "../../models/checkpoint.pth"
WORKING_BEST_MODEL_PATH = "../../models/best_checkpoint.pth"

DATASET_PATH = "../../datasets/Quran_ds/Quran_ds/audio/audio/"
DATASET_PATH_1 = "../../datasets/Quran_ds/Quran_ds/audio/audio/"
TRAIN_DS_PATH = "../../datasets/Quran_ds/quran_train.csv"
TEST_DS_PATH = "../../datasets/Quran_ds/quran_test.csv"


# ? this is for Kaggle training

# MODEL_PATH = "/kaggle/input/datasets/muhammadbannan/quran-ds-v4/best_checkpoint.pth"
# WORKING_MODEL_PATH = "/kaggle/working/checkpoint.pth"
# WORKING_BEST_MODEL_PATH = "/kaggle/working/best_checkpoint.pth"

# DATASET_PATH = "/kaggle/input/datasets/omartariq612/quran-reciters/audio/audio/"
# DATASET_PATH_1 = "/kaggle/input/datasets/abdo3id/female-quran-recitation"

# TRAIN_DS_PATH = "/kaggle/input/datasets/mohammeddeebjalab1/quran-ds-csv-files/quran_train.csv"
# TEST_DS_PATH = "/kaggle/input/datasets/mohammeddeebjalab1/quran-ds-csv-files/quran_test.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 1
SR = 16000
NUM_EPOCHS = 8

DEVICE

device(type='cuda')

In [3]:
BLANK_TOKEN = "<blank>"
SILENT_TOKEN = "<sil>"

IKFAA_LETTERS = [
    "sˤ",
    "ð",
    "θ",
    "k",
    "j",
    "ʃ",
    "q",
    "s",
    "d",
    "tˤ",
    "z",
    "f",
    "t",
    "dˤ",
    "ðˤ",
]

QALQALAA_LETTERS = ["q", "tˤ", "b", "j", "d"]

SHORT_VOWELS: List[str] = ["a", "i", "u"]
LONG_VOWELS: List[str] = ["aa", "ii", "uu"]
TANWEEN: List[str] = ["an", "in", "un"]

SPECIAL_PHONEMES_FOR_TAJWEED = ["n" + l for l in IKFAA_LETTERS]
SPECIAL_PHONEMES_FOR_TAJWEED += ["an" + l for l in IKFAA_LETTERS]
SPECIAL_PHONEMES_FOR_TAJWEED += ["in" + l for l in IKFAA_LETTERS]
SPECIAL_PHONEMES_FOR_TAJWEED += ["un" + l for l in IKFAA_LETTERS]
SPECIAL_PHONEMES_FOR_TAJWEED += [l + "K" for l in QALQALAA_LETTERS]
SPECIAL_PHONEMES_FOR_TAJWEED += ["nn", "mm", "yy", "ww", "rM"]


BASE_CONSONANTS: List[str] = [
    "ʔ",
    "b",
    "t",
    "θ",
    "j",
    "ħ",
    "x",
    "d",
    "ð",
    "r",
    "z",
    "s",
    "ʃ",
    "sˤ",
    "dˤ",
    "tˤ",
    "ðˤ",
    "ʕ",
    "ɣ",
    "f",
    "q",
    "k",
    "l",
    "m",
    "n",
    "h",
    "w",
    "y",
    "T",
]


# Generate all CV combinations
CV_TOKENS = [
    c + v for c in BASE_CONSONANTS for v in SHORT_VOWELS + LONG_VOWELS + TANWEEN
]

# ================================
# All phonemes (for tokenization)
# ================================

PHONEMES: List[str] = (
    [
        SILENT_TOKEN,
    ]
    + SPECIAL_PHONEMES_FOR_TAJWEED
    + BASE_CONSONANTS
    + SHORT_VOWELS
    + LONG_VOWELS
    + TANWEEN
    + CV_TOKENS
)


PHONEMES_CTC: List[str] = [BLANK_TOKEN] + PHONEMES


phoneme_to_id: Dict[str, int] = {p: i for i, p in enumerate(PHONEMES_CTC)}
blank_id: int = phoneme_to_id[BLANK_TOKEN]

In [4]:
len(phoneme_to_id)

371

In [5]:
@contextlib.contextmanager
def suppress_c_stderr():
    """Redirect C-level stderr to /dev/null — catches libmpg123 warnings."""
    with open(os.devnull, "w") as devnull:
        old_fd = os.dup(2)
        os.dup2(devnull.fileno(), 2)
        try:
            yield
        finally:
            os.dup2(old_fd, 2)
            os.close(old_fd)


def add_noise(signal, noise_level=0.003):
    return signal + noise_level * np.random.randn(len(signal))


def random_gain(signal):
    return signal * np.random.uniform(0.8, 1.2)


def load_waveform(audio_path, sr=16000, training=True):

    # audio_path = audio_path.replace(".wav", ".mp3")

    signal = None

    # ── Step 1: try torchaudio — fast, no temp files ──────────────────────
    try:

        waveform, orig_sr = torchaudio.load(audio_path)

        # Stereo → mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample if needed
        if orig_sr != sr:
            waveform = T.Resample(orig_freq=orig_sr, new_freq=sr)(waveform)

        signal = waveform.squeeze(0).numpy()

    # ── Step 2: fallback to librosa for corrupt MP3 frames ────────────────
    except Exception:
        try:
            with suppress_c_stderr():
                signal, _ = librosa.load(audio_path, sr=sr, mono=True)
        except Exception as e:
            raise RuntimeError(f"Both loaders failed for: {audio_path} — {e}")

    # ── Step 3: guard against silent/empty output ─────────────────────────
    if signal is None or len(signal) == 0:
        raise RuntimeError(f"Audio is empty: {audio_path}")

    if np.max(np.abs(signal)) < 1e-6:
        raise RuntimeError(f"Audio appears silent or corrupt: {audio_path}")

    # ── Step 4: augmentation (training only) ──────────────────────────────
    if training and np.random.rand() < 0.5:
        signal = add_noise(signal)

    if training and np.random.rand() < 0.5:
        signal = random_gain(signal)

    # ── Step 5: normalize to [-1, 1] ──────────────────────────────────────
    max_val = np.max(np.abs(signal)) + 1e-8
    signal = signal / max_val

    return torch.tensor(signal, dtype=torch.float32)

In [6]:
class DynamicBatchSampler(torch.utils.data.Sampler):
    """
    Groups samples by audio length so each batch contains
    similarly-lengthed ayahs — minimizes padding waste and OOM risk.
    """

    def __init__(self, dataset, max_samples_per_batch, shuffle=True):
        self.dataset = dataset
        self.max_samples_per_batch = max_samples_per_batch
        self.shuffle = shuffle

        # Pre-read waveform lengths from the dataframe
        # (avoids loading audio just to know the length)
        print("Building length index...")
        self.lengths = []
        for idx in range(len(dataset)):
            row = dataset.df.iloc[idx]

            audio_path = ""
            if row["ds_index"] == 1:
                audio_path = os.path.join(dataset.dataset_path, row["path_of_audio"])
            else:
                audio_path = os.path.join(dataset.dataset_path_1, row["path_of_audio"])

            try:
                info = torchaudio.info(audio_path)
                # Resample length if needed
                length = int(info.num_frames * SR / info.sample_rate)
            except Exception:
                length = SR * 10  # fallback: assume 10 seconds
            self.lengths.append(length)
        print(f"Length index built for {len(self.lengths)} samples")

    def __iter__(self):
        batches = self._build_batches()
        for batch in batches:
            yield batch

    def _build_batches(self):
        import random

        indices = list(range(len(self.lengths)))
        if self.shuffle:
            random.shuffle(indices)
        indices.sort(key=lambda i: self.lengths[i])

        batches = []
        current_batch = []
        current_max_len = 0

        for idx in indices:
            length = self.lengths[idx]
            new_max = max(current_max_len, length)
            if (
                current_batch
                and (len(current_batch) + 1) * new_max > self.max_samples_per_batch
            ):
                batches.append(current_batch)
                current_batch = [idx]
                current_max_len = length
            else:
                current_batch.append(idx)
                current_max_len = new_max

        if current_batch:
            batches.append(current_batch)

        if self.shuffle:
            random.shuffle(batches)

        return batches

    def __len__(self):
        # Build once with no shuffle to get stable count
        indices = list(range(len(self.lengths)))
        indices.sort(key=lambda i: self.lengths[i])
        batches = []
        current_batch = []
        current_max_len = 0
        for idx in indices:
            length = self.lengths[idx]
            new_max = max(current_max_len, length)
            if (
                current_batch
                and (len(current_batch) + 1) * new_max > self.max_samples_per_batch
            ):
                batches.append(current_batch)
                current_batch = [idx]
                current_max_len = length
            else:
                current_batch.append(idx)
                current_max_len = new_max
        if current_batch:
            batches.append(current_batch)
        return len(batches)

In [7]:
class TajweedCTCDataset(Dataset):

    def __init__(
        self, dataframe, dataset_path, dataset_path_1, phoneme_to_id, training=True
    ):
        self.df = dataframe
        self.dataset_path = dataset_path
        self.dataset_path_1 = dataset_path_1
        self.phoneme_to_id = phoneme_to_id
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        audio_path = ""
        if row["ds_index"] == 1:
            audio_path = os.path.join(self.dataset_path, row["path_of_audio"])
        else:
            audio_path = os.path.join(self.dataset_path_1, row["path_of_audio"])

        waveform = load_waveform(
            audio_path,
            training=self.training,
        )

        phoneme_seq = ast.literal_eval(row["phonemes"])
        target_ids = [
            self.phoneme_to_id[p] for p in phoneme_seq if p in self.phoneme_to_id
        ]

        return (
            waveform,
            torch.tensor(target_ids, dtype=torch.long),
            waveform.shape[0],
            len(target_ids),
        )

In [8]:
def ctc_collate(batch):
    waveforms, targets, input_lengths, target_lengths = zip(*batch)
    padded_waveforms = pad_sequence(waveforms, batch_first=True)
    return (
        padded_waveforms,
        torch.cat(targets),
        torch.tensor(input_lengths, dtype=torch.long),
        torch.tensor(target_lengths, dtype=torch.long),
    )

In [9]:
class SpecAugment(nn.Module):
    """
    Conservative SpecAugment tuned for Tajweed:
    - Small time masks to preserve madd duration information
    - Small freq masks to preserve emphatic/nasal phoneme signatures
    - Multiple masks instead of one large one
    """

    def __init__(
        self,
        time_mask_max=8,  # max 80ms erased — safe for short vowels (~150ms)
        freq_mask_max=10,  # max 10/128 channels — preserves spectral shape
        num_time_masks=2,  # two small time masks instead of one big one
        num_freq_masks=2,  # two small freq masks
    ):
        super().__init__()
        self.time_mask_max = time_mask_max
        self.freq_mask_max = freq_mask_max
        self.num_time_masks = num_time_masks
        self.num_freq_masks = num_freq_masks

    def forward(self, x):
        # x: (B, T, C)
        if not self.training:
            return x

        B, T, C = x.shape
        x = x.clone()

        # Multiple small time masks
        for _ in range(self.num_time_masks):
            t = np.random.randint(1, self.time_mask_max + 1)
            t0 = np.random.randint(0, max(1, T - t))
            x[:, t0 : t0 + t, :] = 0

        # Multiple small frequency masks
        for _ in range(self.num_freq_masks):
            f = np.random.randint(1, self.freq_mask_max + 1)
            f0 = np.random.randint(0, max(1, C - f))
            x[:, :, f0 : f0 + f] = 0

        return x

In [10]:
class PhonemeHead(nn.Module):

    def __init__(self, hidden_dim, vocab_size):
        super().__init__()

        self.classifier = nn.Linear(hidden_dim, vocab_size)

    def forward(self, hidden):

        return self.classifier(hidden)

In [11]:
wav2vec2_model = Wav2Vec2Model.from_pretrained(
    "facebook/wav2vec2-large-xlsr-53", ignore_mismatched_sizes=True
)

# Freeze everything
for param in wav2vec2_model.parameters():
    param.requires_grad = True

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-large-xlsr-53 and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
class ASRModel(torch.nn.Module):

    def __init__(self, wav2vec2, spec_augment):
        super().__init__()
        self.wav2vec2 = wav2vec2
        self.conformer = Conformer(
            input_dim=512,
            num_heads=8,
            ffn_dim=2048,
            num_layers=4,
            depthwise_conv_kernel_size=31,
            dropout=0.1,
        )
        self.classifier = PhonemeHead(
            hidden_dim=512,
            vocab_size=len(phoneme_to_id),
        )

        self.spec_augment = spec_augment

    def forward(self, waveforms, input_lengths):

        features = self.wav2vec2.feature_extractor(waveforms)
        features = features.transpose(1, 2)
        feat_lengths = self.wav2vec2._get_feat_extract_output_lengths(input_lengths)

        if self.training:
            features = self.spec_augment(features)

        hidden, hidden_lengths = self.conformer(
            features,
            feat_lengths,
        )

        logits = self.classifier(hidden)
        logits = logits.transpose(0, 1)

        return logits, hidden_lengths 

In [13]:
train_df = pd.read_csv(TRAIN_DS_PATH)
val_df = pd.read_csv(TEST_DS_PATH)

In [14]:
train_df = train_df.head(1)

In [15]:
train_dataset = TajweedCTCDataset(
    dataframe=train_df,
    training=True,
    dataset_path=DATASET_PATH,
    dataset_path_1=DATASET_PATH_1,
    phoneme_to_id=phoneme_to_id,
)
val_dataset = TajweedCTCDataset(
    dataframe=val_df,
    training=False,
    dataset_path=DATASET_PATH,
    dataset_path_1=DATASET_PATH_1,
    phoneme_to_id=phoneme_to_id,
)


# MAX_TOKENS = batch_size * 16000 * 20

# train_sampler = DynamicBatchSampler(
#     train_dataset,
#     max_samples_per_batch=MAX_TOKENS,
#     shuffle=True,
# )

# val_sampler = DynamicBatchSampler(
#     val_dataset,
#     max_samples_per_batch=MAX_TOKENS,
#     shuffle=False,
# )

train_loader = DataLoader(
    train_dataset,
    # batch_sampler=train_sampler,
    batch_size=BATCH_SIZE,
    collate_fn=ctc_collate,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    # batch_sampler=val_sampler,
    batch_size=BATCH_SIZE,
    collate_fn=ctc_collate,
    num_workers=2,
    pin_memory=True,
)

In [16]:
next(enumerate(train_loader))

(0,
 [tensor([[ 0.0000,  0.0000,  0.0000,  ..., -0.0010, -0.0010, -0.0012]]),
  tensor([  1, 293, 313, 328, 195, 175, 110,  67, 326,  72, 301, 308, 318,  96,
          338, 344, 128,  63, 317, 111,  67, 326, 292, 313, 121, 329, 344, 326,
           89, 308, 317, 110,   8, 290,  66, 227, 173,  62, 128, 329, 344, 326,
          304, 326, 263, 308,  99, 338, 318, 326,  84, 221, 336, 177,  96,   1]),
  tensor([297169]),
  tensor([56])])

In [17]:
def save_checkpoint(
    model,
    ctc_loss,
    optimizer,
    epoch,
    # loss,
    best_val_loss,
    epochs_no_improve,
    warmup_scheduler,
    plateau_scheduler,
    path,
):
    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "ctc_loss_state": ctc_loss.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "warmup_scheduler_state": warmup_scheduler.state_dict(),
            "plateau_scheduler_state": plateau_scheduler.state_dict(),
            # "loss": loss,
            "best_val_loss": best_val_loss,
            "epochs_no_improve": epochs_no_improve,
        },
        path,
    )


def load_checkpoint(path):
    checkpoint = torch.load(path)
    return checkpoint

In [18]:
ddp_kwargs = DistributedDataParallelKwargs(find_unused_parameters=True)
accelerator = Accelerator(mixed_precision="fp16", kwargs_handlers=[ddp_kwargs])

print(f"Using device: {DEVICE}")
print(f"Num processes: {accelerator.num_processes}")


ctc_loss = torch.nn.CTCLoss(blank=blank_id, zero_infinity=True, reduction="mean").to(DEVICE)

model = ASRModel(wav2vec2_model, SpecAugment()).to(DEVICE)


WARMUP_EPOCHS = 5
TARGET_CONFORMER_LR = 1e-4  
TARGET_WAV2VEC2_LR = 1e-6

optimizer = torch.optim.AdamW(
    [
        {"params": model.wav2vec2.parameters(), "lr": TARGET_WAV2VEC2_LR},
        {"params": model.conformer.parameters(), "lr": TARGET_CONFORMER_LR},
        {"params": model.classifier.parameters(), "lr": TARGET_CONFORMER_LR}
    ],
    weight_decay=0.01,
)


# Linear warmup for first WARMUP_EPOCHS, then hand off to ReduceLROnPlateau
def warmup_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS  # 0.2, 0.4, 0.6, 0.8, 1.0
    return 1.0  # after warmup, LR stays at target — plateau scheduler takes over


warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_lambda)

plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=1e-9,
)

# ── Class-balance weight tensor ────────────────────────────────────────────
# Since you commented out the frequency-based weights, use uniform weights.
# This is a safe no-op — you can replace it later with real frequency weights.
weight_tensor = torch.ones(len(phoneme_to_id), device=DEVICE)

# Let accelerate handle everything
model, optimizer, train_loader, val_loader = accelerator.prepare(
    model, optimizer, train_loader, val_loader
)


best_val_loss = float("inf")
epochs_no_improve = 0
start_epoch = 0

if os.path.exists(MODEL_PATH):
    print("Loading checkpoint...")
    ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
    accelerator.unwrap_model(model).load_state_dict(ckpt["model_state"])

    optimizer.load_state_dict(ckpt["optimizer_state"])

    # Safe load — handles both old and new checkpoint formats
    if "warmup_scheduler_state" in ckpt:
        warmup_scheduler.load_state_dict(ckpt["warmup_scheduler_state"])
        print("Warmup scheduler state restored ✅")
    else:
        print("⚠️ No warmup scheduler state found — starting fresh (old checkpoint)")

    if "plateau_scheduler_state" in ckpt:
        plateau_scheduler.load_state_dict(ckpt["plateau_scheduler_state"])
        print("Plateau scheduler state restored ✅")
    elif "scheduler_state" in ckpt:
        # Old checkpoint had a single scheduler — load into plateau scheduler
        plateau_scheduler.load_state_dict(ckpt["scheduler_state"])
        print("Plateau scheduler restored from old checkpoint format ✅")
    else:
        print("⚠️ No plateau scheduler state found — starting fresh")

    best_val_loss = ckpt["best_val_loss"]
    epochs_no_improve = ckpt["epochs_no_improve"]
    start_epoch = ckpt["epoch"] + 1
    print(f"Resumed from epoch {start_epoch}")
    print(f"Best val loss so far: {best_val_loss:.4f}")
    print("done loading")
else:
    print("Starting fresh training")
    best_val_loss = float("inf")
    epochs_no_improve = 0
    start_epoch = 0

Using device: cuda
Num processes: 1
Starting fresh training


In [19]:
# audio_path = f"../../datasets/Quran_ds/Quran_ds/audio/audio/{train_df.iloc[0]['path_of_audio']}"

# waveforms, sr = torchaudio.load(audio_path)
# input_lengths = torch.tensor([waveforms.shape[1]], dtype=torch.long)
# input_lengths = input_lengths.to(DEVICE)
# waveforms = waveforms.to(DEVICE)
# logits, lengths = model(waveforms, input_lengths)

# print("Waveforms:", waveforms.shape)
# print("Input lengths:", input_lengths)

# print("Logits:", logits.shape)
# print("Output lengths:", lengths)

# print("Expected max length:", logits.shape[0])
# print("Returned max length:", lengths.max())

In [20]:
model

ASRModel(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection):

In [21]:
def ctc_decode(frame_preds, blank_id):

    decoded = []
    prev = None

    for p in frame_preds:

        p = int(p)

        if p != blank_id and p != prev:
            decoded.append(p)

        prev = p

    return decoded

In [22]:
def compute_per(logits, targets, feat_lengths, target_lengths, blank_id):
    """Phoneme Error Rate — lower is better."""
    pred = torch.argmax(logits, dim=2).permute(1, 0).cpu()  # (B, T)
    feat_lengths = feat_lengths.cpu()
    target_lengths = target_lengths.cpu()
    targets = targets.cpu()

    total_errors = 0
    total_phonemes = 0
    offset = 0

    for i in range(pred.shape[0]):
        decoded = ctc_decode(pred[i, : feat_lengths[i]].tolist(), blank_id)
        length = target_lengths[i].item()
        ref = targets[offset : offset + length].tolist()
        offset += length

        # Simple edit distance
        n, m = len(ref), len(decoded)
        dp = list(range(m + 1))
        for r in ref:
            new_dp = [dp[0] + 1]
            for j, h in enumerate(decoded):
                new_dp.append(
                    min(dp[j + 1] + 1, new_dp[-1] + 1, dp[j] + (0 if r == h else 1))
                )
            dp = new_dp
        total_errors += dp[m]
        total_phonemes += n

    return total_errors / max(total_phonemes, 1)

In [23]:
def compute_blank_rate(logits, feat_lengths, blank_id):
    """Fraction of frames predicted as blank — should drop as penalty takes effect."""
    preds = torch.argmax(logits, dim=2).permute(1, 0)  # (B, T)
    total_frames = 0
    blank_frames = 0
    for i in range(preds.shape[0]):
        frames = preds[i, : feat_lengths[i]]
        blank_frames += (frames == blank_id).sum().item()
        total_frames += feat_lengths[i].item()
    return blank_frames / max(total_frames, 1)

In [24]:
def prepare_targets(targets, target_lengths, blank=0):
    batch_size = targets.size(0)

    prepared = []
    prepared_lengths = []

    for b in range(batch_size):
        length = target_lengths[b].item()
        target = targets[b, :length]

        prev = target[0].item()
        new_target = [prev]

        for symbol_tensor in target[1:]:
            symbol = symbol_tensor.item()

            if symbol == prev:
                new_target.append(blank)

            new_target.append(symbol)
            prev = symbol

        prepared.append(new_target)
        prepared_lengths.append(len(new_target))

    max_len = max(prepared_lengths)

    prepared_tensor = torch.full(
        (batch_size, max_len),
        blank,
        dtype=targets.dtype,
    )

    for b, seq in enumerate(prepared):
        prepared_tensor[b, : len(seq)] = torch.tensor(seq, dtype=targets.dtype)

    prepared_lengths = torch.tensor(prepared_lengths, dtype=target_lengths.dtype)

    return prepared_tensor.to(targets.device), prepared_lengths.to(targets.device)


def ctc_loss_custom(
    log_probs: torch.Tensor,
    targets: torch.Tensor,
    input_lengths: torch.Tensor,
    target_lengths: torch.Tensor,
    blank: int = 0,
    reduction: str = "none",
    finfo_min_fp32: float = torch.finfo(torch.float32).min,
    finfo_min_fp16: float = torch.finfo(torch.float16).min,
) -> torch.Tensor:

    device = log_probs.device

    targets = targets.to(device)
    input_lengths = input_lengths.to(device)
    target_lengths = target_lengths.to(device)

    if targets.dim() == 1:
        targets = targets.unsqueeze(0)

    input_time_size, batch_size = log_probs.shape[:2]
    B = torch.arange(batch_size, device=input_lengths.device)

    targets, target_lengths = prepare_targets(
        targets,
        target_lengths,
        blank,
    )

    zero_padding, zero = 2, torch.tensor(
        finfo_min_fp16 if log_probs.dtype == torch.float16 else finfo_min_fp32,
        device=log_probs.device,
        dtype=log_probs.dtype,
    )

    log_probs_ = log_probs.gather(-1, targets.expand(input_time_size, -1, -1))

    log_alpha = torch.full(
        (input_time_size, batch_size, zero_padding + targets.shape[-1]),
        zero,
        device=log_probs.device,
        dtype=log_probs.dtype,
    )

    log_alpha[0, :, zero_padding] = log_probs[0, B, targets[:, 0]]

    for t in range(1, input_time_size):
        log_alpha[t, :, 2:] = log_probs_[t] + torch.logsumexp(
            torch.stack(
                [
                    log_alpha[t - 1, :, 2:],  # stay
                    log_alpha[t - 1, :, 1:-1],  # move
                ]
            ),
            dim=0,
        )

    last_state = zero_padding + target_lengths - 1

    loss = -log_alpha[
        input_lengths - 1,
        B,
        last_state,
    ]

    if reduction == "mean":
        loss = loss / target_lengths.to(loss.dtype)
        return loss.mean()
    elif reduction == "sum":
        return loss.sum()
    return loss

In [25]:
print("training is starting .......")

training is starting .......


In [26]:
id_to_phoneme = {v: k for k, v in phoneme_to_id.items()}

def custom_ctc_decode(frame_preds, blank_id):
    decoded = []
    prev = None
    count = 0

    for p in frame_preds:
        p = int(p)

        if p == blank_id:
            # ✅ Reset prev so identical phonemes across a blank are treated separately
            if prev is not None and prev != blank_id:
                if decoded:
                    decoded[-1]["count"] = count
                count = 0
            prev = blank_id
            continue

        if p != prev:
            # Save previous phoneme count
            if decoded and prev != blank_id:
                decoded[-1]["count"] = count
            decoded.append({"char": p, "count": 0})
            count = 0

        prev = p
        count += 1

    # Save last phoneme
    if decoded:
        decoded[-1]["count"] = count

    return decoded

In [27]:
def train_model(
    model,
    train_loader,
    val_loader,
    # ctc_loss,
    optimizer,
    warmup_scheduler,
    plateau_scheduler,
    accelerator,
    epochs=30,
    patience=6,
    best_val_loss=float("inf"),
    epochs_no_improve=0,
    warmup_epochs=5,
    start_epoch=0,
):
    for epoch in range(start_epoch, start_epoch + epochs):

        # ── Training loop ──────────────────────────────────────────────
        model.train()
        train_loss = 0
        for waveforms, targets, input_lengths, target_lengths in tqdm(train_loader):

            optimizer.zero_grad(set_to_none=True)

            logits, hidden_len = model(
                waveforms.to(DEVICE),
                torch.tensor([waveforms.shape[1]], dtype=torch.long).to(DEVICE),
            )

            log_probs = torch.log_softmax(logits, dim=-1)

            loss = ctc_loss_custom(
                log_probs,
                targets,
                hidden_len,
                target_lengths,
                reduction="mean",
                blank=blank_id,
            )

            # loss = ctc_loss(log_probs, targets, hidden_len, target_lengths)

            accelerator.backward(loss)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        # ── Validation loop ────────────────────────────────────────────
        model.eval()
        val_loss = 0
        per = 0
        total_blank_rate = 0

        # with torch.no_grad():
        #     for waveforms, targets, input_lengths, target_lengths in tqdm(val_loader):
        #         feat_lengths = get_feature_lengths(input_lengths)

        #         logits = model(waveforms, input_lengths, feat_lengths)
        #         # logits = logits + torch.log(weight_tensor).to(logits.device)

        #         # ✅ Same here — raw logits into ctc_loss
        #         # loss = ctc_loss(logits, targets, feat_lengths, target_lengths)

        #         log_probs = torch.log_softmax(logits, dim=-1)

        #         loss = ctc_loss_custom(
        #             log_probs,
        #             targets,
        #             feat_lengths,
        #             target_lengths,
        #             reduction="mean",
        #             blank=blank_id,
        #         )
        #         val_loss += loss.item()

        #         per += compute_per(
        #             logits, targets, feat_lengths, target_lengths, blank_id
        #         )

        #         # ✅ Track blank rate to verify penalty is working
        #         total_blank_rate += compute_blank_rate(logits, feat_lengths, blank_id)

        # avg_val_loss = val_loss / len(val_loader)
        # per = per / len(val_loader)
        # avg_blank_rate = total_blank_rate / len(val_loader)

        # ── Scheduler step ─────────────────────────────────────────────
        warmup_scheduler.step()
        plateau_scheduler.step(avg_train_loss)

        # ── Logging ────────────────────────────────────────────────────
        print(f"\nEpoch {epoch+1}/{start_epoch + epochs}")
        print(f"  Train Loss   : {avg_train_loss:.4f}")
        # print(f"  Val Loss     : {avg_val_loss:.4f}")
        print(f"  Val PER      : {per:.4f}")
        # print(f"  Blank Rate   : {avg_blank_rate:.2%}  ← target: ~40-50%")
        for group in optimizer.param_groups:
            print(f"Learning rate for group: {group['lr']}")

        print("-" * 40)
        preds = torch.argmax(logits, dim=2).permute(1, 0).cpu()
        decoded_preds = custom_ctc_decode(preds[0], blank_id)
        pred_phonemes_with_frames_count = [
            {"phoneme": id_to_phoneme[int(p["char"])], "count": p["count"]}
            for p in decoded_preds
        ]

        print(pred_phonemes_with_frames_count)

        # ── Checkpointing ──────────────────────────────────────────────
        # unwrapped = accelerator.unwrap_model(model)
        # if avg_val_loss < best_val_loss:
        #     best_val_loss = avg_val_loss
        #     epochs_no_improve = 0
        #     save_checkpoint(
        #         unwrapped,
        #       # unwrapped_loss,
        #         optimizer,
        #         epoch,
        #         avg_val_loss,
        #         best_val_loss,
        #         epochs_no_improve,
        #         warmup_scheduler,
        #         plateau_scheduler,
        #         WORKING_BEST_MODEL_PATH,
        #     )

        #     print("✅ Best model saved")
        # else:
        #     epochs_no_improve += 1
        #     if epochs_no_improve >= patience:
        #         print(f"⏹ Early stopping at epoch {epoch+1}")
        #         break

        # save_checkpoint(
        #     unwrapped,
        #     # unwrapped_loss,
        #     optimizer,
        #     epoch,
        #     avg_val_loss,
        #     best_val_loss,
        #     epochs_no_improve,
        #     warmup_scheduler,
        #     plateau_scheduler,
        #     WORKING_MODEL_PATH,
        # )

In [28]:
train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    # ctc_loss=ctc_loss,
    optimizer=optimizer,
    warmup_scheduler=warmup_scheduler,
    plateau_scheduler=plateau_scheduler,
    accelerator=accelerator,
    epochs=100,  # num_epochs,
    patience=6,
    best_val_loss=best_val_loss,
    epochs_no_improve=epochs_no_improve,
    warmup_epochs=WARMUP_EPOCHS,
    start_epoch=start_epoch,
)

  0%|          | 0/1 [00:00<?, ?it/s]


Epoch 1/100
  Train Loss   : 88.2941
  Val PER      : 0.0000
Learning rate for group: 4e-07
Learning rate for group: 4e-05
Learning rate for group: 4e-05
----------------------------------------
[{'phoneme': 'zi', 'count': 5}, {'phoneme': 'intˤ', 'count': 1}, {'phoneme': 'zi', 'count': 1}, {'phoneme': 'intˤ', 'count': 1}, {'phoneme': 'ʃii', 'count': 1}, {'phoneme': 'zin', 'count': 1}, {'phoneme': 'zi', 'count': 1}, {'phoneme': 'zin', 'count': 1}, {'phoneme': 'and', 'count': 1}, {'phoneme': 'jii', 'count': 1}, {'phoneme': 'zi', 'count': 2}, {'phoneme': 'ðin', 'count': 1}, {'phoneme': 'anð', 'count': 1}, {'phoneme': 'jii', 'count': 1}, {'phoneme': 'ind', 'count': 1}, {'phoneme': 'ðˤin', 'count': 1}, {'phoneme': 'anðˤ', 'count': 1}, {'phoneme': 'zi', 'count': 1}, {'phoneme': 'ðin', 'count': 1}, {'phoneme': 'and', 'count': 1}, {'phoneme': 'ya', 'count': 1}, {'phoneme': 'ðin', 'count': 1}, {'phoneme': 'zi', 'count': 1}, {'phoneme': 'ðin', 'count': 1}, {'phoneme': 'mi', 'count': 1}, {'phone

/home/mahmoud-bannan/miniconda3/envs/torch_mnb/lib/python3.9/site-packages/torch/autograd/graph.py:744: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


  0%|          | 0/1 [00:00<?, ?it/s]


Epoch 2/100
  Train Loss   : 87.1836
  Val PER      : 0.0000
Learning rate for group: 6e-07
Learning rate for group: 6e-05
Learning rate for group: 6e-05
----------------------------------------
[{'phoneme': 'ns', 'count': 1}, {'phoneme': 'ʔuu', 'count': 1}, {'phoneme': 'ns', 'count': 1}, {'phoneme': 'Tuu', 'count': 1}, {'phoneme': 'and', 'count': 1}, {'phoneme': 'mi', 'count': 1}, {'phoneme': 'intˤ', 'count': 1}, {'phoneme': 'nt', 'count': 1}, {'phoneme': 'antˤ', 'count': 1}, {'phoneme': 'sˤa', 'count': 1}, {'phoneme': 'mi', 'count': 1}, {'phoneme': 'ðan', 'count': 1}, {'phoneme': 'zi', 'count': 1}, {'phoneme': 'nt', 'count': 1}, {'phoneme': 'zin', 'count': 1}, {'phoneme': 'ni', 'count': 1}, {'phoneme': 'ħun', 'count': 1}, {'phoneme': 'zi', 'count': 1}, {'phoneme': 'jii', 'count': 1}, {'phoneme': 'xun', 'count': 1}, {'phoneme': 'ʃu', 'count': 1}, {'phoneme': 'ʔii', 'count': 1}, {'phoneme': 'dun', 'count': 1}, {'phoneme': 'zin', 'count': 1}, {'phoneme': 'uns', 'count': 1}, {'phoneme':

...


Epoch 98/100
  Train Loss   : 0.0662
  Val PER      : 0.0000
Learning rate for group: 1e-06
Learning rate for group: 0.0001
Learning rate for group: 0.0001
----------------------------------------
[{'phoneme': '<sil>', 'count': 1}, {'phoneme': 'qaa', 'count': 1}, {'phoneme': 'luu', 'count': 1}, {'phoneme': 'nu', 'count': 1}, {'phoneme': 'rii', 'count': 1}, {'phoneme': 'du', 'count': 1}, {'phoneme': 'ʔa', 'count': 1}, {'phoneme': 'nn', 'count': 1}, {'phoneme': 'na', 'count': 1}, {'phoneme': 'ʔ', 'count': 1}, {'phoneme': 'ku', 'count': 1}, {'phoneme': 'la', 'count': 1}, {'phoneme': 'mi', 'count': 1}, {'phoneme': 'n', 'count': 1}, {'phoneme': 'haa', 'count': 1}, {'phoneme': 'wa', 'count': 1}, {'phoneme': 'ta', 'count': 1}, {'phoneme': 'tˤK', 'count': 1}, {'phoneme': 'ma', 'count': 1}, {'phoneme': 'ʔi', 'count': 1}, {'phoneme': 'nn', 'count': 1}, {'phoneme': 'na', 'count': 1}, {'phoneme': 'qu', 'count': 1}, {'phoneme': 'luu', 'count': 1}, {'phoneme': 'bu', 'count': 1}, {'phoneme': 'naa', 

  0%|          | 0/1 [00:00<?, ?it/s]


Epoch 99/100
  Train Loss   : 0.0568
  Val PER      : 0.0000
Learning rate for group: 1e-06
Learning rate for group: 0.0001
Learning rate for group: 0.0001
----------------------------------------
[{'phoneme': '<sil>', 'count': 1}, {'phoneme': 'qaa', 'count': 1}, {'phoneme': 'luu', 'count': 1}, {'phoneme': 'nu', 'count': 1}, {'phoneme': 'rii', 'count': 1}, {'phoneme': 'du', 'count': 1}, {'phoneme': 'ʔa', 'count': 1}, {'phoneme': 'nn', 'count': 1}, {'phoneme': 'na', 'count': 1}, {'phoneme': 'ʔ', 'count': 1}, {'phoneme': 'ku', 'count': 1}, {'phoneme': 'la', 'count': 1}, {'phoneme': 'mi', 'count': 1}, {'phoneme': 'n', 'count': 1}, {'phoneme': 'haa', 'count': 1}, {'phoneme': 'wa', 'count': 1}, {'phoneme': 'ta', 'count': 1}, {'phoneme': 'tˤK', 'count': 1}, {'phoneme': 'ma', 'count': 1}, {'phoneme': 'ʔi', 'count': 1}, {'phoneme': 'nn', 'count': 1}, {'phoneme': 'na', 'count': 1}, {'phoneme': 'qu', 'count': 1}, {'phoneme': 'luu', 'count': 1}, {'phoneme': 'bu', 'count': 1}, {'phoneme': 'naa', 

  0%|          | 0/1 [00:00<?, ?it/s]


Epoch 100/100
  Train Loss   : 0.0515
  Val PER      : 0.0000
Learning rate for group: 1e-06
Learning rate for group: 0.0001
Learning rate for group: 0.0001
----------------------------------------
[{'phoneme': '<sil>', 'count': 1}, {'phoneme': 'qaa', 'count': 1}, {'phoneme': 'luu', 'count': 1}, {'phoneme': 'nu', 'count': 1}, {'phoneme': 'rii', 'count': 1}, {'phoneme': 'du', 'count': 1}, {'phoneme': 'ʔa', 'count': 1}, {'phoneme': 'nn', 'count': 1}, {'phoneme': 'na', 'count': 1}, {'phoneme': 'ʔ', 'count': 1}, {'phoneme': 'ku', 'count': 1}, {'phoneme': 'la', 'count': 1}, {'phoneme': 'mi', 'count': 1}, {'phoneme': 'n', 'count': 1}, {'phoneme': 'haa', 'count': 1}, {'phoneme': 'wa', 'count': 1}, {'phoneme': 'ta', 'count': 1}, {'phoneme': 'tˤK', 'count': 1}, {'phoneme': 'ma', 'count': 1}, {'phoneme': 'ʔi', 'count': 1}, {'phoneme': 'nn', 'count': 1}, {'phoneme': 'na', 'count': 1}, {'phoneme': 'qu', 'count': 1}, {'phoneme': 'luu', 'count': 1}, {'phoneme': 'bu', 'count': 1}, {'phoneme': 'naa',

In [29]:
# from collections import Counter
# import ast

# counter = Counter()

# for phonemes in train_df["phonemes"]:
#     seq = ast.literal_eval(phonemes)
#     counter.update(seq)

# print(counter["<sil>"])
# print(counter.most_common())

In [30]:
# def predict_phonemes(path: str, device):
#     waveforms = load_waveform(path, training=False)
#     waveforms = waveforms.unsqueeze(0).to(device)
#     model.eval()
#     with torch.no_grad():
#         input_lengths = torch.tensor(
#             [waveforms.shape[1]], dtype=torch.long, device=device
#         )
#         feat_lengths = get_feature_lengths(input_lengths)
#         logits = model(waveforms, input_lengths, feat_lengths)
#         preds = torch.argmax(logits, dim=2).permute(1, 0).cpu()

#         decoded_preds = custom_ctc_decode(preds[0], blank_id)
#         pred_phonemes = [id_to_phoneme[int(p["char"])] for p in decoded_preds]
#         pred_phonemes_with_frames_count = [
#             {"phoneme": id_to_phoneme[int(p["char"])], "count": p["count"]}
#             for p in decoded_preds
#         ]
#     return pred_phonemes, pred_phonemes_with_frames_count

In [31]:
# item = train_df.iloc[-1]

# audio_path = audio_path = os.path.join(DATASET_PATH, item["path_of_audio"])
# phonemes = item["phonemes"]
# phonemes

In [32]:
# audio_path

In [33]:
# model.eval()
# with torch.no_grad():
#     predicted_phonemes, pred_phonemes_with_frames_count = predict_phonemes(
#         audio_path,
#         DEVICE,
#     )

#     print(predicted_phonemes)
#     print()
#     print(pred_phonemes_with_frames_count)

In [34]:
# count = 0

# model.eval()
# with torch.no_grad():

#     for row in train_df.itertuples():
#         audio_path = row.path_of_audio
#         if row.ds_index == 1:
#             audio_path = os.path.join(DATASET_PATH, audio_path)
#         else:
#             audio_path = os.path.join(DATASET_PATH_1, audio_path)

#         predicted_phonemes = predict_phonemes(
#             audio_path,
#             device,
#         )

#         if "<sil>" in predicted_phonemes[1:-1]:
#             count += 1

# print(count)

In [35]:
# print(count)

In [36]:
# import torch
# import numpy as np
# from sklearn.metrics import confusion_matrix
# import matplotlib.pyplot as plt
# import seaborn as sns

# start = 0

# def ctc_greedy_decode(logits, blank_id):
#     """
#     logits: [T, V]
#     returns: list of token ids
#     """
#     probs = torch.softmax(logits, dim=-1)
#     pred_ids = torch.argmax(probs, dim=-1).tolist()

#     result = []
#     prev = None

#     for p in pred_ids:
#         if p != blank_id and p != prev:
#             result.append(p)
#         prev = p

#     return result


# model.eval()

# blank_id = phoneme_to_id["<blank>"]  # adjust if different

# y_true = []
# y_pred = []

# with torch.no_grad():
#     for waveforms, targets, input_lengths, target_lengths in tqdm(val_loader):

#         feat_lengths = get_feature_lengths(input_lengths)
#         logits = model(waveforms, input_lengths, feat_lengths)

#         for b in range(logits.size(1)):

#             logit_seq = logits[:, b, :]  # [T, V]

#             pred_ids = ctc_greedy_decode(logit_seq, blank_id)

#             # -------------------------
#             # FIX TARGET HANDLING
#             # -------------------------
#             L = target_lengths[b].item()

#             true_ids = targets[start:start + L].tolist()
#             start += L

#             y_pred.extend(pred_ids)
#             y_true.extend(true_ids)

# num_classes = len(phoneme_to_id)

# cm = confusion_matrix(
#     y_true,
#     y_pred,
#     labels=list(range(num_classes))
# )


# labels = [id_to_phoneme[i] for i in range(num_classes)]

# plt.figure(figsize=(20, 16))
# sns.heatmap(cm, xticklabels=False, yticklabels=False)
# plt.title("Phoneme Confusion Matrix")
# plt.show()